# Wide schemas: staying complete when the field count is large

Ask a model to fill one big schema in a single call and the answers get worse as the schema
grows. The output budget goes to brackets and quotes instead of values, and a wide schema can
overflow the context window before it finishes. nfield splits the schema into groups that fit
the model, extracts each, and reassembles the nested JSON you asked for.

This notebook does two things: it estimates the work for a very wide schema without calling any
model, then runs a real extraction on a smaller wide schema so you can see the completeness
numbers.

## Setup

```bash
pip install "nfield[groq,cli]"
export GROQ_API_KEY="gsk_..."
```

In [1]:
import json
import os
import subprocess
import tempfile

from nfield import NField

assert os.environ.get("GROQ_API_KEY"), "set GROQ_API_KEY first"

MODEL = "groq/llama-3.3-70b-versatile"

## Estimate first, no API calls

`nfield inspect` reads a schema and reports the field count, the type breakdown, and a
minimum-call estimate (`K_min`). It never contacts a provider, so it is free and instant. Here
we build a 2000-field schema on the fly and inspect it.

In [2]:
wide_schema = {
    "type": "object",
    "properties": {f"field_{i:04d}": {"type": "string"} for i in range(2000)},
}

path = os.path.join(tempfile.mkdtemp(), "wide_schema.json")
with open(path, "w") as handle:
    json.dump(wide_schema, handle)

# max-output-tokens is the model's real output ceiling; K_min scales with it.
report = subprocess.run(
    ["nfield", "inspect", path, "--max-output-tokens", "32768"],
    capture_output=True,
    text=True,
)
print(report.stdout or report.stderr)

Schema: C:\Users\Dhana\AppData\Local\Temp\tmplkmv4_ce\wide_schema.json
Total leaf fields : 2000
K_min estimate    : 4  (M_O=32768, English NSL)
Field types:
  string     2000
Paths:
  field_0000
  field_0001
  field_0002
  field_0003
  field_0004
  field_0005
  field_0006
  field_0007
  field_0008
  field_0009
  field_0010
  field_0011
  field_0012
  field_0013
  field_0014
  field_0015
  field_0016
  field_0017
  field_0018
  field_0019
  ... and 1980 more



`K_min` is a lower bound from the output budget alone: even 2000 fields fit in a handful of calls,
not one call per field. A real run may use a few more calls to keep each one small and reliable,
which is why the benchmark runs in the README report higher counts for thousands of fields. Either
way the call count grows slowly as the schema widens, instead of the whole thing failing in one
overflowing request.

## Run a real wide extraction

Now a smaller wide schema against a real document. We reuse one `NField` engine so the schema is
parsed and the model calibrated once. Passing the model's real `context_window` and
`max_output_tokens` lets nfield plan across the full window.

In [3]:
document = """
ACME ROBOTICS - EQUIPMENT DATASHEET

Model: AR-9 Titan
Serial: TR-2024-88213
Category: industrial arm
Payload: 12 kg
Reach: 1300 mm
Axes: 6
Repeatability: 0.02 mm
Weight: 240 kg
Mounting: floor
Power: 3-phase 400V
Peak power draw: 4.5 kW
Controller: AR-CTRL-7
Firmware: 5.2.1
IP rating: IP67
Operating temp: 0 to 45 C
Warranty: 24 months
Manufacturer: Acme Robotics
Country of origin: Germany
Release year: 2024
"""

schema = {
    "type": "object",
    "properties": {
        "model": {"type": "string"},
        "serial": {"type": "string"},
        "category": {"type": "string"},
        "payload_kg": {"type": "number"},
        "reach_mm": {"type": "number"},
        "axes": {"type": "integer"},
        "repeatability_mm": {"type": "number"},
        "weight_kg": {"type": "number"},
        "mounting": {"type": "string"},
        "power": {"type": "string"},
        "peak_power_kw": {"type": "number"},
        "controller": {"type": "string"},
        "firmware": {"type": "string"},
        "ip_rating": {"type": "string"},
        "operating_temp": {"type": "string"},
        "warranty_months": {"type": "integer"},
        "manufacturer": {"type": "string"},
        "country_of_origin": {"type": "string"},
        "release_year": {"type": "integer"},
    },
}

engine = NField(MODEL, schema, context_window=131_072, max_output_tokens=32_768)
result = engine.extract(document)
result.data

{'model': 'AR-9 Titan',
 'serial': 'TR-2024-88213',
 'category': 'industrial arm',
 'payload_kg': 12.0,
 'reach_mm': 1300.0,
 'axes': 6,
 'repeatability_mm': 0.02,
 'weight_kg': 240.0,
 'mounting': 'floor',
 'power': '3-phase 400V',
 'peak_power_kw': 4.5,
 'controller': 'AR-CTRL-7',
 'firmware': '5.2.1',
 'ip_rating': 'IP67',
 'operating_temp': '0 to 45 C',
 'warranty_months': 24,
 'manufacturer': 'Acme Robotics',
 'country_of_origin': 'Germany',
 'release_year': 2024}

In [4]:
meta = result.metadata
print("status           :", result.status.value)
print("fields extracted :", meta.fields_extracted, "/", meta.fields_total)
print("calls (K / K_min):", meta.K, "/", meta.K_min)
print("quality score    :", round(meta.quality_score, 3))

status           : success
fields extracted : 19 / 19
calls (K / K_min): 1 / 1
quality score    : 1.0


The fields come back complete, and the call count stays close to the computed minimum. That is
the whole idea: split only as much as the budget requires, then reassemble.